# GranthStudio v3.1 — Linux CPU verification (no GPU)

Runs the accepted CPU baseline on Linux and writes an evidence bundle. **Session settings:** accelerator **None** (CPU),
Internet **On** only if pip must install a missing test dependency. The dataset `granthstudio-cpu-baseline` (git bundle,
see docs/KAGGLE_RUNBOOK.md) must be attached. Nothing here downloads a model or uses a GPU. The verdict printed at the
end is computed from what actually ran; do not copy a PASS from anywhere else.

In [ ]:
import glob, json, os, platform, shutil, subprocess, sys
BASELINE = "f8d5d02f609a68806e8272617d1e04c98cf6040c"   # accepted CPU baseline (Codex-reviewed)
bundles = glob.glob("/kaggle/input/**/granthstudio.bundle", recursive=True)
assert bundles, "attach the granthstudio-cpu-baseline dataset (granthstudio.bundle) to this notebook"
BUNDLE = bundles[0]
print("python", sys.version.split()[0], "| platform", platform.platform(), "| bundle", BUNDLE)
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout if shutil.which("nvidia-smi") else "no nvidia-smi: CPU session (expected)")

In [ ]:
WORK = "/kaggle/working"
def run(*cmd, cwd=None):
    done = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print("$", " ".join(cmd), "->", done.returncode); print((done.stdout + done.stderr).strip()[-1500:])
    assert done.returncode == 0
    return done.stdout.strip()
shutil.rmtree(f"{WORK}/harness_repo", ignore_errors=True); shutil.rmtree(f"{WORK}/granthstudio", ignore_errors=True)
run("git", "clone", "-q", "-b", "sprint/remaining-pipeline", BUNDLE, f"{WORK}/harness_repo")                 # branch tip: contains the harness scripts
HARNESS_SHA = run("git", "rev-parse", "HEAD", cwd=f"{WORK}/harness_repo")
shutil.copytree(f"{WORK}/harness_repo/scripts/kaggle", f"{WORK}/harness")   # copy before the code under test is checked out
run("git", "clone", "-q", "-b", "sprint/remaining-pipeline", BUNDLE, f"{WORK}/granthstudio")
run("git", "checkout", "-q", "--detach", BASELINE, cwd=f"{WORK}/granthstudio")   # the exact accepted commit
print("code under test:", run("git", "rev-parse", "HEAD", cwd=f"{WORK}/granthstudio"), "| harness:", HARNESS_SHA)

In [ ]:
done = subprocess.run([sys.executable, f"{WORK}/harness/linux_cpu_verify.py", "--repo", f"{WORK}/granthstudio",
    "--harness-dir", f"{WORK}/harness", "--baseline", BASELINE, "--harness-sha", HARNESS_SHA, "--out", f"{WORK}/linux_cpu_evidence"],
    capture_output=True, text=True)
print(done.stdout[-3000:]); print(done.stderr[-2000:])

In [ ]:
receipt = json.load(open(f"{WORK}/linux_cpu_evidence/receipt.json"))
print("VERDICT:", receipt["verdict"], receipt.get("reason", ""))
ev = receipt.get("evaluation", {})
print("counts:", ev.get("counts")); print("problems:", ev.get("problems"))
print("skipped:"); [print("  ", s) for s in ev.get("skipped", [])]
print("required POSIX tests:"); [print("  ", v, k) for k, v in ev.get("required", {}).items()]
print("supplementary:", ev.get("supplementary"))
print("\nDownload: /kaggle/working/linux_cpu_evidence.zip (or the folder). Keep this notebook version and its output with it.")